# Lesson 13A: Exploration Strategies - Theory

## Introduction

Every algorithm so far used epsilon-greedy or entropy bonuses for exploration — simple, but wasteful: epsilon-greedy explores *uniformly at random*, ignoring everything already learned about which actions look promising. This lesson covers exploration strategies that use that information.

- **UCB**: explore in proportion to uncertainty, via a confidence bound
- **Thompson sampling**: explore by sampling from a posterior belief over rewards
- **Intrinsic motivation**: manufacture a reward signal for *novelty itself*, for environments where extrinsic reward is too sparse to learn from at all
- **Random Network Distillation**: a practical, scalable way to measure "have I seen something like this before?"

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

## Part 1: The Exploration-Exploitation Dilemma

The cleanest setting to study exploration in isolation is the **multi-armed bandit**: $k$ actions ("arms"), each with an unknown reward distribution, no state transitions. Every pull teaches you about one arm's payoff while costing you the chance to pull whichever arm is actually best.

**Regret** formalizes the cost of exploration:
$$\text{Regret}(T) = \sum_{t=1}^T \left(\mu^* - \mu_{a_t}\right)$$

where $\mu^*$ is the best arm's true mean and $a_t$ is the arm pulled at step $t$. A good algorithm's regret grows **sub-linearly** in $T$ — it eventually settles almost always on the best arm. Uniform random exploration (or a fixed-$\epsilon$ epsilon-greedy) grows regret **linearly**, because it keeps exploring at a constant rate forever, long after there's anything left to learn.

## Part 2 & 3: UCB and Thompson Sampling

### UCB (Upper Confidence Bound)

**Optimism in the face of uncertainty**: rank each arm by an upper confidence bound on its mean, not the mean itself. Arms tried rarely get a big bonus (we might be wrong about them); arms tried often get almost no bonus (we're probably right about them).

$$a_t = \arg\max_a \left[\hat{\mu}_a + \sqrt{\frac{2 \ln t}{N_a}}\right]$$

$\hat{\mu}_a$ is the empirical mean reward for arm $a$, $N_a$ is how many times it's been pulled. The bonus term shrinks as $N_a$ grows (Hoeffding's inequality) and grows (slowly, via $\ln t$) as time passes if an arm keeps being neglected — it can never be ignored forever.

### Thompson Sampling

A Bayesian alternative: maintain a **posterior distribution** over each arm's mean, sample one value from each posterior, and pull whichever arm's sample is highest. For Bernoulli rewards, the conjugate prior is a Beta distribution — updating it after observing a reward is a one-line increment.

$$\text{arm}_a \sim \text{Beta}(\alpha_a, \beta_a) \quad\to\quad a_t = \arg\max_a \text{arm}_a$$

After pulling arm $a$ and observing reward $r \in \{0, 1\}$: $\alpha_a \mathrel{+}= r$, $\beta_a \mathrel{+}= (1 - r)$. Arms with little data have wide, uncertain posteriors — sampling naturally explores them sometimes; arms with lots of data have narrow posteriors concentrated near their true mean.

In [ ]:
k = 10
true_means = np.random.uniform(0.2, 0.8, k)
best_mean = true_means.max()
T = 2000
n_runs = 50  # average over runs for a smooth regret curve


def run_epsilon_greedy(epsilon=0.1):
    counts, values = np.zeros(k), np.zeros(k)
    regret = np.zeros(T)
    for t in range(1, T + 1):
        if np.random.rand() < epsilon or np.any(counts == 0):
            untried = np.where(counts == 0)[0]
            a = np.random.choice(untried) if len(untried) else np.random.randint(k)
        else:
            a = np.argmax(values)
        r = float(np.random.rand() < true_means[a])
        counts[a] += 1
        values[a] += (r - values[a]) / counts[a]
        regret[t - 1] = best_mean - true_means[a]
    return np.cumsum(regret)


def run_ucb1():
    counts, values = np.zeros(k), np.zeros(k)
    regret = np.zeros(T)
    for t in range(1, T + 1):
        if np.any(counts == 0):
            a = np.argmin(counts)
        else:
            bonus = np.sqrt(2 * np.log(t) / counts)
            a = np.argmax(values + bonus)
        r = float(np.random.rand() < true_means[a])
        counts[a] += 1
        values[a] += (r - values[a]) / counts[a]
        regret[t - 1] = best_mean - true_means[a]
    return np.cumsum(regret)


def run_thompson():
    alpha, beta_p = np.ones(k), np.ones(k)
    regret = np.zeros(T)
    for t in range(T):
        a = np.argmax(np.random.beta(alpha, beta_p))
        r = float(np.random.rand() < true_means[a])
        alpha[a] += r
        beta_p[a] += (1 - r)
        regret[t] = best_mean - true_means[a]
    return np.cumsum(regret)


regret_curves = {'epsilon-greedy (0.1)': [], 'UCB1': [], 'Thompson sampling': []}
for _ in range(n_runs):
    regret_curves['epsilon-greedy (0.1)'].append(run_epsilon_greedy())
    regret_curves['UCB1'].append(run_ucb1())
    regret_curves['Thompson sampling'].append(run_thompson())

fig, ax = plt.subplots(figsize=(8, 5))
for name, curves in regret_curves.items():
    mean_curve = np.mean(curves, axis=0)
    ax.plot(mean_curve, label=f'{name} (final: {mean_curve[-1]:.1f})')
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative regret')
ax.set_title(f'Regret on a {k}-armed bandit, averaged over {n_runs} runs')
ax.legend()
plt.tight_layout()
plt.show()

## Part 4: Intrinsic Motivation, Curiosity, and Count-Based Exploration

UCB and Thompson sampling assume every action gets *some* reward signal. In hard-exploration problems (sparse-reward mazes, Montezuma's Revenge) the agent can wander for thousands of steps without a single nonzero reward — there's nothing for UCB's confidence bound or Thompson's posterior to update on. The fix: manufacture reward from **novelty itself**.

### Count-Based Exploration

Add a bonus inversely proportional to how often a state has been visited:
$$r^+(s) = r(s) + \frac{\beta}{\sqrt{N(s)}}$$

Exact counts $N(s)$ only work for small discrete state spaces. **Pseudo-counts** (Bellemare et al., 2016) generalize this to large/continuous spaces using a density model's probability estimate as a proxy for "how many times have I effectively seen something like this."

### Curiosity (Intrinsic Curiosity Module, ICM)

Train a forward model to predict the next state's *features* from the current state and action; use the model's **prediction error** as the intrinsic reward:
$$r^i_t = \eta \, \lVert \hat{\phi}(s_{t+1}) - \phi(s_{t+1}) \rVert^2$$

States the model already predicts well are boring (low reward); poorly-predicted states are surprising, and get explored more. The subtlety: predict in a learned *feature* space $\phi(s)$, not raw pixels — otherwise the agent is rewarded for finding stochastic noise it can never learn to predict (the "noisy TV problem").

## Part 5: Random Network Distillation (RND)

RND (Burda et al., 2018) sidesteps the noisy-TV problem entirely and scales cleanly to complex observations:

1. A **fixed, randomly-initialized target network** $f: s \to \mathbb{R}^d$ — never trained, just a fixed random projection
2. A **trainable predictor network** $\hat{f}: s \to \mathbb{R}^d$, trained to match the target's output on states the agent has actually visited
3. Intrinsic reward = prediction error: $r^i(s) = \lVert \hat{f}(s) - f(s) \rVert^2$

Why this works: the predictor can only get good at matching the target on states it's been *trained on* — i.e., states the agent has actually visited. Novel states get high prediction error by construction, and (crucially) that error doesn't depend on whether the environment itself is stochastic — the target function is deterministic even if the environment isn't, so there's no noisy-TV trap.

In [ ]:
state_dim = 4


class RandomNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 32), nn.ReLU(), nn.Linear(32, 16))

    def forward(self, x):
        return self.net(x)


target_net = RandomNet()
for param in target_net.parameters():
    param.requires_grad = False  # fixed forever, by design

predictor_net = RandomNet()
optimizer = optim.Adam(predictor_net.parameters(), lr=1e-3)


def novelty(states):
    with torch.no_grad():
        target_out = target_net(states)
    pred_out = predictor_net(states)
    return ((pred_out - target_out) ** 2).mean(dim=-1)


# "Familiar" states: a small cluster the agent has visited repeatedly.
# "Novel" state: something far outside that cluster, never encountered.
familiar_states = torch.randn(5, state_dim)
novel_state = torch.randn(1, state_dim) * 3 + 10

print("Novelty BEFORE training the predictor on familiar states:")
print(f"  familiar (mean): {novelty(familiar_states).mean().item():.4f}")
print(f"  novel:           {novelty(novel_state).item():.4f}")

# Simulate repeated visits: train the predictor only on the familiar states.
for _ in range(300):
    target_out = target_net(familiar_states).detach()
    pred_out = predictor_net(familiar_states)
    loss = nn.functional.mse_loss(pred_out, target_out)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\nNovelty AFTER training the predictor on familiar states:")
print(f"  familiar (mean): {novelty(familiar_states).mean().item():.4f}")
print(f"  novel:           {novelty(novel_state).item():.4f}")
print("\nThe predictor learns to match the target on visited states -- their novelty")
print("bonus collapses toward zero -- while the never-seen state's error stays high.")

## Key Concepts

1. **Regret**: the formal cost of exploration — a good bandit algorithm's regret grows sub-linearly in $T$
2. **UCB**: optimism in the face of uncertainty — explore arms in proportion to a shrinking confidence bound
3. **Thompson sampling**: maintain a posterior per arm, sample and act greedily on the sample — often the best empirical performer despite UCB's cleaner theoretical guarantees
4. **Count-based bonuses**: reward $\propto 1/\sqrt{N(s)}$; pseudo-counts extend this beyond small discrete spaces
5. **Curiosity (ICM)**: reward = forward-model prediction error in a learned feature space — avoid raw pixels to dodge the noisy-TV problem
6. **RND**: reward = a trainable predictor's error against a *fixed random* target network — no noisy-TV trap, scales to high-dimensional observations, and is simple enough to bolt onto almost any existing agent